In [21]:
import pandas as pd
import numpy as np

SHEET_ID = "1mAI15Yq5w5Yl7XHEYrFih04q4THkOLGUww9wJ5PZx8M"
GID = "60520972"  # aba Página1
url_tsv = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=tsv&gid={GID}"

ttest_data_m = pd.read_csv(url_tsv, sep='\t')
train_data_m.head()

print("Colunas lidas:", list(train_data_m.columns))
display(train_data_m.head())

Colunas lidas: ['Idade', 'Diagnóstico', 'Astigmatismo', 'Taxa lacrimal', 'Tipo de lente']


,Idade,Diagnóstico,Astigmatismo,Taxa lacrimal,Tipo de lente
0,jovem,miopia,não,reduzida,nenhuma
1,jovem,miopia,sim,normal,gelatinosa
2,jovem,hipermetropia,não,normal,gelatinosa
3,jovem,hipermetropia,sim,normal,dura
4,pré-presbiopia,miopia,não,reduzida,gelatinosa


In [22]:
import pandas as pd
import numpy as np

SHEET_ID = "1mAI15Yq5w5Yl7XHEYrFih04q4THkOLGUww9wJ5PZx8M"
GID = "60520972"
url_tsv = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=tsv&gid={GID}"

train_data_m = pd.read_csv(url_tsv, sep="\t")

print("Colunas lidas:", list(train_data_m.columns))
display(train_data_m.head())


def calc_total_entropy(train_data, label, class_list):
    total_row = train_data.shape[0]
    total_entr = 0.0

    for c in class_list:
        total_class_count = train_data[train_data[label] == c].shape[0]
        if total_class_count != 0:
            p = total_class_count / total_row
            total_entr += -p * np.log2(p)

    return total_entr


def calc_entropy(feature_value_data, label, class_list):
    class_count = feature_value_data.shape[0]
    entropy = 0.0

    for c in class_list:
        label_class_count = feature_value_data[feature_value_data[label] == c].shape[0]
        if label_class_count != 0:
            p = label_class_count / class_count
            entropy += -p * np.log2(p)

    return entropy


def calc_info_gain(feature_name, train_data, label, class_list, verbose=False):
    feature_value_list = train_data[feature_name].unique()
    total_row = train_data.shape[0]
    feature_info = 0.0

    if verbose:
        print(f"\nAtributo: {feature_name}")

    for feature_value in feature_value_list:
        feature_value_data = train_data[train_data[feature_name] == feature_value]
        feature_value_count = feature_value_data.shape[0]
        feature_value_entropy = calc_entropy(feature_value_data, label, class_list)
        feature_value_probability = feature_value_count / total_row
        feature_info += feature_value_probability * feature_value_entropy

        if verbose:
            classes = feature_value_data[label].value_counts().to_dict()
            print(
                f"  Valor = {feature_value} | "
                f"total = {feature_value_count} | "
                f"classes = {classes} | "
                f"entropia = {feature_value_entropy:.2f}"
            )

    total_entropy = calc_total_entropy(train_data, label, class_list)
    gain = total_entropy - feature_info

    if verbose:
        print(f"  Entropia do conjunto: {total_entropy:.2f}")
        print(f"  Entropia ponderada: {feature_info:.2f}")
        print(f"  Ganho de informação: {gain:.2f}")

    return gain


def find_most_informative_feature(train_data, label, class_list):
    feature_list = train_data.columns.drop(label)

    if len(feature_list) == 0:
        return None

    max_info_gain = -1
    max_info_feature = None

    for feature in feature_list:
        feature_info_gain = calc_info_gain(feature, train_data, label, class_list)
        if feature_info_gain > max_info_gain:
            max_info_gain = feature_info_gain
            max_info_feature = feature

    return max_info_feature


def generate_sub_tree(feature_name, train_data, label, class_list):
    feature_value_count_dict = train_data[feature_name].value_counts(sort=False)
    tree = {}

    for feature_value, count in feature_value_count_dict.items():
        feature_value_data = train_data[train_data[feature_name] == feature_value]

        assigned_to_node = False
        for c in class_list:
            class_count = feature_value_data[feature_value_data[label] == c].shape[0]

            if class_count == count:
                tree[feature_value] = c
                assigned_to_node = True
                break

        if not assigned_to_node:
            tree[feature_value] = "?"

    return tree


def majority_class(train_data, label):
    return train_data[label].value_counts().idxmax()


def make_tree(root, prev_feature_value, train_data, label, class_list):
    if train_data.shape[0] == 0:
        return

    if len(train_data[label].unique()) == 1:
        if prev_feature_value is not None:
            root[prev_feature_value] = train_data[label].iloc[0]
        return

    max_info_feature = find_most_informative_feature(train_data, label, class_list)

    if max_info_feature is None:
        if prev_feature_value is not None:
            root[prev_feature_value] = majority_class(train_data, label)
        return

    tree = generate_sub_tree(max_info_feature, train_data, label, class_list)

    if prev_feature_value is not None:
        root[prev_feature_value] = {max_info_feature: tree}
        next_root = root[prev_feature_value][max_info_feature]
    else:
        root[max_info_feature] = tree
        next_root = root[max_info_feature]

    for node, branch in list(next_root.items()):
        if branch == "?":
            feature_value_data = train_data[train_data[max_info_feature] == node].copy()
            feature_value_data = feature_value_data.drop(columns=[max_info_feature])
            make_tree(next_root, node, feature_value_data, label, class_list)


def id3(train_data_m, label):
    train_data = train_data_m.copy()
    tree = {}
    class_list = train_data[label].unique()
    make_tree(tree, None, train_data, label, class_list)
    return tree


def predict(tree, instance):
    if not isinstance(tree, dict):
        return tree

    root_node = next(iter(tree))
    feature_value = instance[root_node]

    if feature_value in tree[root_node]:
        return predict(tree[root_node][feature_value], instance)
    return None


def evaluate(tree, test_data_m, label):
    correct_predict = 0
    wrong_predict = 0

    for index, row in test_data_m.iterrows():
        result = predict(tree, test_data_m.iloc[index])
        if result == test_data_m[label].iloc[index]:
            correct_predict += 1
        else:
            wrong_predict += 1

    accuracy = correct_predict / (correct_predict + wrong_predict)
    return accuracy


def print_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(indent + "->", tree)
        return

    for key, value in tree.items():
        print(indent + str(key))
        if isinstance(value, dict):
            for subkey, subvalue in value.items():
                print(indent + f"  [{subkey}]")
                print_tree(subvalue, indent + "    ")
        else:
            print(indent + "  ->", value)


label = "Tipo de lente"
class_list = train_data_m[label].unique()

print("\n" + "="*50)
print("PARTE (a) - GANHOS DE INFORMAÇÃO DA RAIZ")
print("="*50)

entropia_total = calc_total_entropy(train_data_m, label, class_list)
print(f"Entropia do conjunto total: {entropia_total:.2f}")

for feature in train_data_m.columns.drop(label):
    calc_info_gain(feature, train_data_m, label, class_list, verbose=True)

raiz = find_most_informative_feature(train_data_m, label, class_list)
print(f"\nAtributo escolhido como raiz: {raiz}")

print("\n" + "="*50)
print("PARTE (b) - RAMO Taxa lacrimal = reduzida")
print("="*50)

sub_reduzida = train_data_m[train_data_m["Taxa lacrimal"] == "reduzida"].copy()
class_list_reduzida = sub_reduzida[label].unique()

print(f"Quantidade de exemplos no ramo: {sub_reduzida.shape[0]}")
print(f"Entropia do ramo: {calc_total_entropy(sub_reduzida, label, class_list_reduzida):.2f}")

for feature in sub_reduzida.columns.drop([label, "Taxa lacrimal"]):
    calc_info_gain(feature, sub_reduzida, label, class_list_reduzida, verbose=True)

proximo_atributo = find_most_informative_feature(
    sub_reduzida.drop(columns=["Taxa lacrimal"]),
    label,
    class_list_reduzida
)
print(f"\nPróximo atributo escolhido no ramo reduzida: {proximo_atributo}")

print("\n" + "="*50)
print("ÁRVORE FINAL ID3")
print("="*50)

tree = id3(train_data_m, label)
print(tree)

print("\nÁrvore formatada:")
print_tree(tree)

test_data_m = pd.read_csv(url_tsv, sep="\t")
accuracy = evaluate(tree, test_data_m, label)

print(f"\nAcurácia: {accuracy:.2f}")

Colunas lidas: ['Idade', 'Diagnóstico', 'Astigmatismo', 'Taxa lacrimal', 'Tipo de lente']


,Idade,Diagnóstico,Astigmatismo,Taxa lacrimal,Tipo de lente
0,jovem,miopia,não,reduzida,nenhuma
1,jovem,miopia,sim,normal,gelatinosa
2,jovem,hipermetropia,não,normal,gelatinosa
3,jovem,hipermetropia,sim,normal,dura
4,pré-presbiopia,miopia,não,reduzida,gelatinosa



PARTE (a) - GANHOS DE INFORMAÇÃO DA RAIZ
Entropia do conjunto total: 1.46

Atributo: Idade
  Valor = jovem | total = 4 | classes = {'gelatinosa': 2, 'nenhuma': 1, 'dura': 1} | entropia = 1.50
  Valor = pré-presbiopia | total = 5 | classes = {'gelatinosa': 2, 'dura': 2, 'nenhuma': 1} | entropia = 1.52
  Valor = presbiopia | total = 6 | classes = {'gelatinosa': 4, 'dura': 1, 'nenhuma': 1} | entropia = 1.25
  Entropia do conjunto: 1.46
  Entropia ponderada: 1.41
  Ganho de informação: 0.05

Atributo: Diagnóstico
  Valor = miopia | total = 8 | classes = {'gelatinosa': 4, 'nenhuma': 2, 'dura': 2} | entropia = 1.50
  Valor = hipermetropia | total = 7 | classes = {'gelatinosa': 4, 'dura': 2, 'nenhuma': 1} | entropia = 1.38
  Entropia do conjunto: 1.46
  Entropia ponderada: 1.44
  Ganho de informação: 0.01

Atributo: Astigmatismo
  Valor = não | total = 8 | classes = {'gelatinosa': 5, 'nenhuma': 2, 'dura': 1} | entropia = 1.30
  Valor = sim | total = 7 | classes = {'gelatinosa': 3, 'dura': 3,